# FASE 2 · MrBERT-es con 600 anotaciones v3

Fine-tuning de [`BSC-LT/MrBERT-es`](https://huggingface.co/BSC-LT/MrBERT-es), un ModernBERT bilingüe español–inglés de 150M parámetros y contexto largo.

**Uso:** en Colab selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU**, y luego **Entorno de ejecución → Ejecutar todas**. La ejecución guarda cada fold en Drive y puede reanudarse. No usa citas, fundamentos, confianza ni estratos como predictores.

Esta evaluación usa los folds históricos de desarrollo; **no es una prueba final independiente**.


In [ ]:
# 1. Comprobar GPU e instalar dependencias reproducibles
import os, subprocess, sys
assert subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0, "Activa una GPU en el entorno de Colab"
!pip -q install "transformers==4.57.6" "accelerate==1.10.1" "sentencepiece==0.2.1" "safetensors==0.8.0" "scikit-learn==1.9.1" "pandas==3.0.5" "openpyxl==3.1.5"
print("Dependencias instaladas")


In [ ]:
# 2. Descargar exactamente la rama de trabajo
import os, shutil, subprocess
REPO = "https://github.com/joako0o/FASE_2.git"
BRANCH = "arena/01a0b014-fase-2"
PROJECT = "/content/FASE_2"
if os.path.exists(PROJECT): shutil.rmtree(PROJECT)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
os.chdir(PROJECT)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", commit)


In [ ]:
# 3. Montar Drive: los folds terminados sobreviven a una desconexión
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/FASE_2/resultados_mrbert_600_v3"
os.makedirs(OUT, exist_ok=True)
print("Salida reanudable:", OUT)


In [ ]:
# 4. Entrenar/evaluar los 5 folds agrupados (puede tardar; es reanudable)
import subprocess, sys
cmd = [sys.executable, "scripts/evaluar_mrbert_600_v3.py", "--salida", OUT, "--folds", "1,2,3,4,5"]
subprocess.run(cmd, check=True)


In [ ]:
# 5. Mostrar métricas y controles básicos
import json, os, pandas as pd
metrics_path = os.path.join(OUT, "metricas.json")
if os.path.exists(metrics_path):
    result = json.load(open(metrics_path, encoding="utf-8"))
    display(pd.DataFrame(result["mrbert_600"]["por_clase"]).T)
    print(json.dumps({k: result["mrbert_600"][k] for k in ["accuracy", "macro_f1", "f1_hd", "errores", "h_d_cruzados", "direccion_a_neutral", "neutral_a_direccion"]}, indent=2))
else:
    print("Aún no están los 5 folds. Vuelve a ejecutar desde la celda 4; los terminados se omiten.")


In [ ]:
# 6. Empaquetar y descargar resultados auditables
import os, shutil
from google.colab import files
if os.path.exists(os.path.join(OUT, "manifest.json")):
    archive = shutil.make_archive("/content/resultados_mrbert_600_v3", "zip", OUT)
    print("ZIP:", archive)
    files.download(archive)
else:
    print("No se genera ZIP hasta completar los cinco folds.")
